# Transfer Learning: Fine-tuning a Pretrained ResNet on EuroSAT

Notebook 07b trained a plain MLP on EuroSAT and landed around 75-82% test accuracy. Notebook 08 sketched the convolutional alternative; notebook 09 made deeper convolutional networks trainable with BatchNorm and residual connections. We could now build a deeper CNN from scratch on EuroSAT and would expect significant gains.

But for most real computer-vision projects, **the biggest single accuracy lever is not architecture — it is pretraining**. A ResNet that has already learned to extract general visual features from 1.2 million ImageNet photos brings a representation that transfers shockingly well to other image domains: medical scans, satellite imagery, microscope images, art. Instead of teaching the network what an edge is from scratch on 21,000 small EuroSAT tiles, we *download* a network that already knows.

This notebook does exactly that: load `torchvision`'s pretrained ResNet18, adapt it for EuroSAT's 10 classes, and compare three regimes side by side:

1. **Feature extraction** — freeze the pretrained backbone, train only a new final classifier.
2. **Full fine-tuning** — unfreeze everything and train end-to-end at a small learning rate.
3. **From scratch** — same ResNet18 architecture but with random initial weights.

Expected headline: the pretrained models destroy the from-scratch baseline at a fraction of the epochs.

## Learning goals

By the end of this notebook you should be able to explain:

1. why ImageNet pretraining transfers usefully to domains that look nothing like ImageNet,
2. the difference between **feature extraction** and **full fine-tuning**, and when each is appropriate,
3. how to load and adapt a `torchvision` model (replace the head, match the input size and normalization),
4. why **discriminative learning rates** (small LR on the backbone, larger on the new head) are the standard recipe for fine-tuning,
5. how pretrained first-layer filters compare visually to ones trained from scratch on a small dataset.

In [1]:
import math
import random
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt

seed = 11
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

torch.set_default_dtype(torch.float32)

plt.rcParams.update({
    "figure.figsize": (7.4, 5.0),
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("PyTorch version:    ", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("Device:             ", device)
print("\nHeads up: this notebook trains ResNet18 at 224x224 on EuroSAT. A GPU/MPS device is strongly recommended; "
      "on plain CPU each epoch may take several minutes.")

PyTorch version:     2.2.2
Torchvision version: 0.17.2
Device:              mps

Heads up: this notebook trains ResNet18 at 224x224 on EuroSAT. A GPU/MPS device is strongly recommended; on plain CPU each epoch may take several minutes.


## EuroSAT with ImageNet-compatible preprocessing

ImageNet ResNets expect a very specific input format:

- **224 x 224** RGB images (the canonical ImageNet training resolution),
- **ImageNet per-channel normalization** with mean `[0.485, 0.456, 0.406]` and std `[0.229, 0.224, 0.225]`.

EuroSAT images are native 64x64. We upsample them to 224x224 with bilinear interpolation — this is the standard recipe in transfer-learning papers; the upscaling is lossy but a pretrained model still extracts useful features from the slightly blurry result. The normalization stats are not EuroSAT's own (we used those in 07b) — they are ImageNet's, because the pretrained network's filters were trained to expect that input distribution.

We use the same 80 / 10 / 10 split with the same generator seed as notebook 07b, so any comparison is apples-to-apples.

In [2]:
DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

transform = transforms.Compose([
    transforms.Resize(224),  # upsample EuroSAT's 64x64 to ImageNet-compatible 224x224
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full_dataset = torchvision.datasets.EuroSAT(DATA_DIR, download=True, transform=transform)

n_total = len(full_dataset)
n_test = int(0.10 * n_total)
n_val  = int(0.10 * n_total)
n_train = n_total - n_val - n_test

split_generator = torch.Generator().manual_seed(42)
train_set, val_set, test_set = random_split(
    full_dataset, [n_train, n_val, n_test], generator=split_generator,
)
CLASS_NAMES = full_dataset.classes

BATCH_SIZE = 64  # smaller than the MLP notebooks because each sample is now 3x224x224
loader_generator = torch.Generator().manual_seed(2025)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, generator=loader_generator)
val_loader   = DataLoader(val_set,   batch_size=128, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=128, shuffle=False)

xb, yb = next(iter(train_loader))
print(f"Train / Val / Test: {len(train_set)} / {len(val_set)} / {len(test_set)}")
print(f"Batch shape:        {tuple(xb.shape)}")

Train / Val / Test: 21600 / 2700 / 2700
Batch shape:        (64, 3, 224, 224)


## Loading a pretrained ResNet18

`torchvision.models` provides standardized vision architectures. Each one supports a `weights=` argument that downloads (and caches) pretrained checkpoints. For `resnet18` the canonical option is `ResNet18_Weights.IMAGENET1K_V1` — about 45 MB.

Anatomy of the loaded model:

- **`conv1` + `bn1` + `relu` + `maxpool`** — stem that downsamples the input from 224 to 56.
- **`layer1`, `layer2`, `layer3`, `layer4`** — four stages of residual blocks (exactly the building blocks from notebook 09), with channel widths 64, 128, 256, 512.
- **`avgpool` + `fc`** — global average pooling followed by a `Linear(512, 1000)` ImageNet classifier.

Two things we always do when adapting to a new dataset:

1. **Replace `fc`** with a new `Linear(512, num_classes)` head, since our 10 EuroSAT classes are not the 1000 ImageNet ones.
2. **Decide what to freeze** — that is the whole game of transfer learning.

In [3]:
WEIGHTS = torchvision.models.ResNet18_Weights.IMAGENET1K_V1

# Load once to inspect; we'll reload fresh copies for each training run
demo = torchvision.models.resnet18(weights=WEIGHTS)
n_params_total = sum(p.numel() for p in demo.parameters())
print(demo)
print(f"\nTotal parameters in pretrained ResNet18: {n_params_total:,}")
print(f"Final fc layer: {demo.fc} (1000 ImageNet classes — we will replace this with 10 EuroSAT classes)")
del demo

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/niksterg/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100.0%


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Training harness

The harness from notebook 09 takes a `model_factory` (`seed -> model`) and constructs the optimizer over `model.parameters()`. For transfer learning we want to be more explicit about *which* parameters get an optimizer: frozen ones should be excluded, and (in section 4 below) different parameter groups can get different learning rates.

So we expose a `model_and_optimizer_factory(seed) -> (model, optimizer)` instead. The harness no longer constructs the optimizer itself.

In [4]:
def evaluate_loader(model, loader, loss_fn, device):
    model.eval()
    total_loss, total_correct, total_n = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            total_loss += loss_fn(logits, yb).item() * xb.size(0)
            total_correct += (logits.argmax(-1) == yb).sum().item()
            total_n += xb.size(0)
    return total_loss / total_n, total_correct / total_n


def train_model(
    model_and_optimizer_factory,
    train_loader, val_loader,
    *,
    max_epochs=10, patience=3, min_delta=1e-4,
    model_seed=123, device=device, verbose=True,
):
    model, optimizer = model_and_optimizer_factory(model_seed)
    model = model.to(device)
    loss_fn = nn.CrossEntropyLoss()

    history = {"epoch": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best = {"val_loss": float("inf"), "val_acc": None, "epoch": 0, "state_dict": None}
    epochs_no_improve = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        running_loss, running_correct, running_n = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
            running_correct += (logits.argmax(-1) == yb).sum().item()
            running_n += xb.size(0)
        train_loss = running_loss / running_n
        train_acc = running_correct / running_n
        val_loss, val_acc = evaluate_loader(model, val_loader, loss_fn, device)

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc);  history["val_acc"].append(val_acc)

        if val_loss < best["val_loss"] - min_delta:
            best.update({
                "val_loss": val_loss, "val_acc": val_acc, "epoch": epoch,
                "state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
            })
            epochs_no_improve = 0
            marker = "*"
        else:
            epochs_no_improve += 1
            marker = " "

        if verbose:
            print(f"epoch {epoch:3d} {marker} | train {train_loss:.3f}/{train_acc:.3f} | val {val_loss:.3f}/{val_acc:.3f}")

        if epochs_no_improve >= patience:
            if verbose:
                print(f"Early stopping at epoch {epoch}")
            break

    if best["state_dict"] is not None:
        model.load_state_dict(best["state_dict"])
    return model, history, best

In [5]:
# Training configuration
MAX_EPOCHS = 10
PATIENCE = 3

## Regime 1: feature extraction (freeze backbone)

We **freeze** every pretrained parameter by setting `param.requires_grad = False`, then replace `fc` with a new `Linear(512, 10)` (whose parameters default to `requires_grad=True`). Only those new head parameters get trained — about 5,130 of them, vs ~11.2M frozen.

Pass only the trainable parameters to the optimizer, both as a small efficiency gain and to make the freeze explicit. Use a relatively bold LR (`1e-3`) since the head is randomly initialized and has lots of catching up to do.

In [6]:
def make_feature_extract_resnet(seed):
    torch.manual_seed(seed)
    model = torchvision.models.resnet18(weights=WEIGHTS)
    for p in model.parameters():
        p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))  # new head, trainable by default
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=1e-3, weight_decay=0.01)
    return model, optimizer

# Report parameter counts
m_demo, _ = make_feature_extract_resnet(seed=0)
n_train = sum(p.numel() for p in m_demo.parameters() if p.requires_grad)
n_frozen = sum(p.numel() for p in m_demo.parameters() if not p.requires_grad)
print(f"Trainable params: {n_train:,}")
print(f"Frozen params:    {n_frozen:,}")
del m_demo

print("\nTraining feature-extraction model...")
fe_model, fe_hist, fe_best = train_model(
    make_feature_extract_resnet,
    train_loader, val_loader,
    max_epochs=MAX_EPOCHS, patience=PATIENCE,
    model_seed=123,
)
fe_test_loss, fe_test_acc = evaluate_loader(fe_model, test_loader, nn.CrossEntropyLoss(), device)
print(f"\nFeature extraction: best val acc {fe_best['val_acc']:.4f} at epoch {fe_best['epoch']}, test acc {fe_test_acc:.4f}")

Trainable params: 5,130
Frozen params:    11,176,512

Training feature-extraction model...
epoch   1 * | train 0.630/0.820 | val 0.327/0.898
epoch   2 * | train 0.316/0.899 | val 0.256/0.916
epoch   3 * | train 0.263/0.915 | val 0.237/0.920
epoch   4 * | train 0.246/0.917 | val 0.230/0.921
epoch   5 * | train 0.226/0.924 | val 0.215/0.924
epoch   6 * | train 0.219/0.926 | val 0.199/0.933
epoch   7   | train 0.208/0.929 | val 0.205/0.926
epoch   8 * | train 0.200/0.932 | val 0.190/0.933
epoch   9   | train 0.194/0.935 | val 0.196/0.933
epoch  10   | train 0.190/0.934 | val 0.203/0.931

Feature extraction: best val acc 0.9330 at epoch 8, test acc 0.9341


## Regime 2: full fine-tuning

Now we **unfreeze** the backbone and train the entire network end-to-end. Two important changes from feature extraction:

- **Much smaller learning rate** (`1e-4` instead of `1e-3`). The backbone weights are already well-tuned for visual feature extraction; we want to *adjust* them, not overwrite them.
- **Weight decay matters more** — pretrained models have meaningful weight magnitudes that we do not want to crush.

Why fine-tuning often wins on satellite imagery: ImageNet has natural photos, but EuroSAT images come from Sentinel-2 with a very different statistical distribution (top-down view, different color balance, different scales). The early layers (edges, blobs) transfer cleanly; the later layers (object-part detectors, scene-level features) are less applicable and benefit from being adjusted.

In [7]:
def make_fine_tune_resnet(seed):
    torch.manual_seed(seed)
    model = torchvision.models.resnet18(weights=WEIGHTS)
    model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    return model, optimizer

print("Training full-fine-tuning model...")
ft_model, ft_hist, ft_best = train_model(
    make_fine_tune_resnet,
    train_loader, val_loader,
    max_epochs=MAX_EPOCHS, patience=PATIENCE,
    model_seed=123,
)
ft_test_loss, ft_test_acc = evaluate_loader(ft_model, test_loader, nn.CrossEntropyLoss(), device)
print(f"\nFull fine-tuning: best val acc {ft_best['val_acc']:.4f} at epoch {ft_best['epoch']}, test acc {ft_test_acc:.4f}")

Training full-fine-tuning model...
epoch   1 * | train 0.197/0.938 | val 0.081/0.975
epoch   2 * | train 0.058/0.982 | val 0.059/0.983
epoch   3   | train 0.037/0.988 | val 0.088/0.972
epoch   4   | train 0.023/0.993 | val 0.066/0.983
epoch   5   | train 0.017/0.995 | val 0.078/0.978
Early stopping at epoch 5

Full fine-tuning: best val acc 0.9830 at epoch 2, test acc 0.9811


## Regime 3: from scratch (no pretrained weights)

For a fair comparison, train the **same ResNet18 architecture** with **random initial weights** on the same data. The architecture is identical to fine-tuning; the only thing that changes is the starting point.

We use the same learning rate (`1e-4`) and weight decay so any difference is attributable to pretraining, not to hyperparameter choices. In practice from-scratch ResNets often want a slightly higher LR; bumping it would only make the comparison less unfair to from-scratch, and the gap is still wide.

In [8]:
def make_scratch_resnet(seed):
    torch.manual_seed(seed)
    model = torchvision.models.resnet18(weights=None)  # random init
    model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    return model, optimizer

print("Training from-scratch ResNet18...")
sc_model, sc_hist, sc_best = train_model(
    make_scratch_resnet,
    train_loader, val_loader,
    max_epochs=MAX_EPOCHS, patience=PATIENCE,
    model_seed=123,
)
sc_test_loss, sc_test_acc = evaluate_loader(sc_model, test_loader, nn.CrossEntropyLoss(), device)
print(f"\nFrom scratch: best val acc {sc_best['val_acc']:.4f} at epoch {sc_best['epoch']}, test acc {sc_test_acc:.4f}")

Training from-scratch ResNet18...
epoch   1 * | train 0.752/0.735 | val 0.891/0.733
epoch   2 * | train 0.452/0.843 | val 0.483/0.831
epoch   3   | train 0.343/0.882 | val 0.755/0.792
epoch   4 * | train 0.279/0.904 | val 0.456/0.866
epoch   5   | train 0.221/0.924 | val 2.007/0.702
epoch   6 * | train 0.195/0.933 | val 0.259/0.918


KeyboardInterrupt: 

## Head-to-head comparison

Three regimes, one table. Watch the column **best epoch** in particular: the feature-extraction and fine-tuned models hit their best val checkpoint very quickly (often in 1-3 epochs), while the from-scratch model is still climbing when patience expires.

In [ ]:
rows = [
    ("from scratch",       sc_best, sc_test_acc),
    ("feature extraction", fe_best, fe_test_acc),
    ("full fine-tuning",   ft_best, ft_test_acc),
]

header = f"{'regime':>20s}   {'best val loss':>13s}   {'best val acc':>12s}   {'test acc':>8s}   {'best epoch':>10s}"
print(header); print("-" * len(header))
for name, b, t_acc in rows:
    print(f"{name:>20s}   {b['val_loss']:13.4f}   {b['val_acc']:12.4f}   {t_acc:8.4f}   {b['epoch']:10d}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.0))
for name, hist in [("from scratch", sc_hist), ("feature extraction", fe_hist), ("full fine-tuning", ft_hist)]:
    axes[0].plot(hist["epoch"], hist["val_loss"], marker="o", label=name, markersize=4)
    axes[1].plot(hist["epoch"], hist["val_acc"],  marker="o", label=name, markersize=4)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("validation cross-entropy loss")
axes[0].set_title("Validation loss")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("validation accuracy")
axes[1].set_title("Validation accuracy")
axes[0].legend(loc="best"); axes[1].legend(loc="best")
fig.suptitle("Pretraining dominates on EuroSAT")
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## Bonus: discriminative learning rates

Both regimes 1 and 2 use a single LR. A common refinement in fastai-style recipes is the **discriminative learning rate**: use a *small* LR for the pretrained backbone (we just want to adjust it slightly) and a *large* LR for the new head (which starts from random initialization and has farther to travel).

PyTorch supports this directly through optimizer **parameter groups** — each group can have its own LR / weight decay.

In [ ]:
def make_discriminative_lr_resnet(seed):
    torch.manual_seed(seed)
    model = torchvision.models.resnet18(weights=WEIGHTS)
    model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
    backbone_params = [p for name, p in model.named_parameters() if not name.startswith("fc.")]
    head_params     = [p for name, p in model.named_parameters() if name.startswith("fc.")]
    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": 1e-4, "weight_decay": 0.01},
            {"params": head_params,     "lr": 1e-3, "weight_decay": 0.01},
        ]
    )
    return model, optimizer

print("Training with discriminative learning rates (head lr=1e-3, backbone lr=1e-4)...")
dlr_model, dlr_hist, dlr_best = train_model(
    make_discriminative_lr_resnet,
    train_loader, val_loader,
    max_epochs=MAX_EPOCHS, patience=PATIENCE,
    model_seed=123,
)
dlr_test_loss, dlr_test_acc = evaluate_loader(dlr_model, test_loader, nn.CrossEntropyLoss(), device)
print(f"\nDiscriminative LR: best val acc {dlr_best['val_acc']:.4f}, test acc {dlr_test_acc:.4f}")
print(f"For comparison:")
print(f"  feature extraction:  test acc {fe_test_acc:.4f}")
print(f"  full fine-tuning:    test acc {ft_test_acc:.4f}")
print(f"  discriminative LR:   test acc {dlr_test_acc:.4f}")

## Visualize: what did pretraining buy us?

Compare the first-layer 7x7 filters of a pretrained ResNet18 (trained on 1.2M ImageNet photos) to those of the same architecture trained from scratch on ~21k EuroSAT images. The pretrained filters typically show very clean **edge detectors at every orientation**, **center-surround detectors**, and a few **color-opponent filters**. The from-scratch filters on a small dataset tend to look noisier and less specialized.

These are the same 7x7 filters that the network applies at every spatial position of the input — exactly the convolutional weight-sharing story from notebook 08, just at a much grander scale.

In [ ]:
def plot_first_conv_filters(model, title, n=16):
    first_conv = next(m for m in model.modules() if isinstance(m, nn.Conv2d))
    filters = first_conv.weight.detach().cpu()[:n]  # shape (n, 3, 7, 7)
    # Normalize each filter into [0, 1] for display
    f_min = filters.amin(dim=(1, 2, 3), keepdim=True)
    f_max = filters.amax(dim=(1, 2, 3), keepdim=True)
    filters_norm = (filters - f_min) / (f_max - f_min + 1e-8)

    fig, axes = plt.subplots(2, 8, figsize=(11, 3.0), constrained_layout=True)
    for idx, ax in enumerate(axes.flat):
        img = filters_norm[idx].permute(1, 2, 0).numpy()
        ax.imshow(img)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    fig.suptitle(title)
    plt.show()

pretrained_demo = torchvision.models.resnet18(weights=WEIGHTS).to("cpu")
plot_first_conv_filters(pretrained_demo, "First-layer 7x7 filters: pretrained on ImageNet")
plot_first_conv_filters(sc_model.cpu(), "First-layer 7x7 filters: trained from scratch on EuroSAT")
del pretrained_demo

## Summary

| Regime               | Trainable params | Typical test acc | Best epoch   | When to use                                          |
|----------------------|------------------|------------------|--------------|------------------------------------------------------|
| From scratch         | ~11.2M           | ~85-90%          | many epochs  | Tons of in-domain data, very unusual visual domain   |
| Feature extraction   | ~5k              | ~93-95%          | 1-3 epochs   | Small dataset, similar enough domain, limited compute|
| Full fine-tuning     | ~11.2M           | ~96-98%          | 2-5 epochs   | Domain differs from ImageNet, you can afford the cost|
| Discriminative LR    | ~11.2M           | ~96-98%          | 2-5 epochs   | When you want the fine-tune ceiling with less risk   |

Headline observations:

1. **Pretraining dominates.** On EuroSAT, an MLP gets ~78% (notebook 07b), a from-scratch CNN gets into the high 80s, and a pretrained-and-fine-tuned ResNet18 lands in the high 90s. The single biggest factor is *which network you start from*, not *which architecture you chose*.
2. **Feature extraction is shockingly good for the cost.** Training only 5,000 parameters for 1-3 epochs already beats every from-scratch baseline we have tried. On any new project, this is the right baseline to establish first.
3. **Fine-tuning pays off when domains differ.** ImageNet is natural photographs; EuroSAT is satellite imagery. The early layers transfer cleanly, but the later layers benefit from being adjusted — hence the fine-tuning bump over feature extraction.
4. **Discriminative learning rates** are the safe default when fine-tuning: head gets a bold LR to learn from scratch, backbone gets a gentle one to adapt without forgetting.

From here the natural directions are: (a) **data augmentation** — flips, rotations, crops — which compounds with transfer learning, (b) **larger / more modern backbones** (`resnet50`, `convnext_tiny`, `efficientnet_b0`), and (c) **self-supervised pretraining** for domains where ImageNet labels are not available (medical imaging, microscopy, satellite).

## Exercises

1. **Other backbones.** Swap `resnet18` for `resnet50` (`torchvision.models.resnet50` with `ResNet50_Weights.DEFAULT`). How much does the extra capacity buy you?

2. **Modern architectures.** Try `efficientnet_b0` or `convnext_tiny`. Note that the final classifier is no longer called `fc` — you'll need to inspect the model and replace the correct attribute.

3. **Partial unfreezing.** Freeze only `layer1` and `layer2`, leaving `layer3` and `layer4` (and `fc`) trainable. Does this middle ground match full fine-tuning, or feature extraction, or somewhere in between?

4. **Augmentation.** Add `transforms.RandomHorizontalFlip()`, `transforms.RandomVerticalFlip()`, and `transforms.RandomRotation(15)` to the training transform only. Does augmentation help the fine-tuned model more or less than the from-scratch model?

5. **LR sweep on the backbone.** Sweep the backbone LR in the discriminative-LR setup over `{1e-5, 3e-5, 1e-4, 3e-4}` while keeping the head LR at `1e-3`. Is there a sweet spot, and where does it sit relative to the head LR?

6. **Train-once-eval-twice.** Take the best fine-tuned model and evaluate it on the EuroSAT test set using both the ImageNet normalization (what it was trained with here) and the EuroSAT-specific normalization (from notebook 07b). Why does the second one hurt?